# **Projectwork 1: Exploring Deep Learning Techniques for Basic Point Cloud Analysis in Environmental Monitoring**

**Specialication:** Digital Environment  
**Authors:** Florian Klaver, Nicola Noger  
**Supervisor:** Martin Schüle  
**Submission Deadline:** 22.05.2025


---
## **Introduction**

Here we shortly describe the project. Not too much detail rest in report

---
## **Setup**

In [17]:
# All imports

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import laspy
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, TensorDataset

# Enable vectorized graphics
%config InlineBackend.figure_formats = ["svg"]

---
## **Preprocessing**

#### Loading the data

In [27]:
path_flo = "C:/Users/flori/Desktop/pa-1_data/zurich_1.las"
# path_nicola = 

las = laspy.read(path_flo)

# List all available attributes in the LAS file
print("Available attributes in LAS file:")
for spec in las.point_format.dimension_names:
    print("-", spec)


# total number of points
n_points = len(las.points)
print("\nTotal number of points:", n_points)

print(las.header)
print(las.points)

Available attributes in LAS file:
- X
- Y
- Z
- intensity
- return_number
- number_of_returns
- scan_direction_flag
- edge_of_flight_line
- classification
- synthetic
- key_point
- withheld
- scan_angle_rank
- user_data
- point_source_id
- gps_time

Total number of points: 12902017
<LasHeader(1.2, <PointFormat(1, 0 bytes of extra dims)>)>
<ScaleAwarePointRecord(fmt: <PointFormat(1, 0 bytes of extra dims)>, len: 12902017, point size: 28)>


In [ ]:
# Create a pandas DataFrame with the LAS data for initial data exploration
df = pd.DataFrame({
    "X": las.x,
    "Y": las.y,
    "Z": las.z,
    "intensity": las.intensity,
    "return_number": las.return_number,
    "number_of_returns": las.number_of_returns,
    "scan_direction_flag": las.scan_direction_flag,
    "edge_of_flight_line": las.edge_of_flight_line,
    "synthetic": las.synthetic,
    "key_point": las.key_point,
    "withheld": las.withheld,
    "user_data": las.user_data,
    "point_source_id": las.point_source_id,
    "scan_angle_rank": las.scan_angle_rank,
    "gps_time": las.gps_time,
    "classification": las.classification
})

# get a first impression of the data
display(df.info())
display(df.head())
display(df.describe())

display("Classification values:")
display(df["classification"].value_counts())


In [ ]:
# Check the data types of all columns
print(df.dtypes)



X                       object
Y                       object
Z                       object
intensity               uint16
return_number           object
number_of_returns       object
scan_direction_flag     object
edge_of_flight_line     object
synthetic               object
key_point               object
withheld                object
user_data                uint8
point_source_id         uint16
scan_angle_rank           int8
gps_time               float64
classification          object
dtype: object


In [29]:
# For columns that should be integers or floats, convert them
df['return_number'] = pd.to_numeric(df['return_number'], errors='coerce')
df['number_of_returns'] = pd.to_numeric(df['number_of_returns'], errors='coerce')


In [ ]:
print(df.dtypes)

X                       object
Y                       object
Z                       object
intensity               uint16
return_number          float64
number_of_returns      float64
scan_direction_flag     object
edge_of_flight_line     object
synthetic               object
key_point               object
withheld                object
user_data                uint8
point_source_id         uint16
scan_angle_rank           int8
gps_time               float64
classification          object
dtype: object


In [20]:
# Identify list columns
list_columns = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, list)).any()]

# Check if we have identified any list columns
if not list_columns:
    print("No list columns detected.")
else:
    print(f"List columns found: {list_columns}")

# Only proceed if list columns are found
if list_columns:
    # Ensure all list columns have equal length per row
    max_len = df[list_columns[0]].apply(len).max()  # Assuming all lists are the same length per row

    # Expand each list into separate rows
    df_expanded = pd.DataFrame({
        col: df[col].explode() if col in list_columns else df[col].repeat(max_len)
        for col in df.columns
    }).reset_index(drop=True)

    display(df_expanded.head())  # Check the result


No list columns detected.


In [6]:
df_exploded = df.explode(["X", "Y", "Z", "return_number", "classification"])

df_exploded.head()


,X,Y,Z,intensity,return_number,number_of_returns,scan_direction_flag,edge_of_flight_line,synthetic,key_point,withheld,user_data,point_source_id,scan_angle_rank,gps_time,classification
0,"[2683033.91, 2683074.91, 2683086.29, 2683135.9...","[1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...","[433.53000000000003, 426.62, 433.13, 417.98, 4...",71,"[1, 1, 1, 2, 1, 2, 1, 2, 2, 1, 1, 1, 1, 1, 2, ...","[1, 2, 1, 2, 1, 3, 1, 3, 2, 1, 1, 1, 1, 1, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",47,125,29,2.051151e+08,"[6, 6, 6, 6, 6, 3, 2, 3, 1, 6, 6, 6, 2, 6, 6, ..."
1,"[2683033.91, 2683074.91, 2683086.29, 2683135.9...","[1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...","[433.53000000000003, 426.62, 433.13, 417.98, 4...",73,"[1, 1, 1, 2, 1, 2, 1, 2, 2, 1, 1, 1, 1, 1, 2, ...","[1, 2, 1, 2, 1, 3, 1, 3, 2, 1, 1, 1, 1, 1, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",46,125,29,2.051151e+08,"[6, 6, 6, 6, 6, 3, 2, 3, 1, 6, 6, 6, 2, 6, 6, ..."
2,"[2683033.91, 2683074.91, 2683086.29, 2683135.9...","[1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...","[433.53000000000003, 426.62, 433.13, 417.98, 4...",114,"[1, 1, 1, 2, 1, 2, 1, 2, 2, 1, 1, 1, 1, 1, 2, ...","[1, 2, 1, 2, 1, 3, 1, 3, 2, 1, 1, 1, 1, 1, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",49,125,29,2.051151e+08,"[6, 6, 6, 6, 6, 3, 2, 3, 1, 6, 6, 6, 2, 6, 6, ..."
3,"[2683033.91, 2683074.91, 2683086.29, 2683135.9...","[1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...","[433.53000000000003, 426.62, 433.13, 417.98, 4...",106,"[1, 1, 1, 2, 1, 2, 1, 2, 2, 1, 1, 1, 1, 1, 2, ...","[1, 2, 1, 2, 1, 3, 1, 3, 2, 1, 1, 1, 1, 1, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",67,125,28,2.051151e+08,"[6, 6, 6, 6, 6, 3, 2, 3, 1, 6, 6, 6, 2, 6, 6, ..."
4,"[2683033.91, 2683074.91, 2683086.29, 2683135.9...","[1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...","[433.53000000000003, 426.62, 433.13, 417.98, 4...",183,"[1, 1, 1, 2, 1, 2, 1, 2, 2, 1, 1, 1, 1, 1, 2, ...","[1, 2, 1, 2, 1, 3, 1, 3, 2, 1, 1, 1, 1, 1, 2, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",50,125,28,2.051151e+08,"[6, 6, 6, 6, 6, 3, 2, 3, 1, 6, 6, 6, 2, 6, 6, ..."


---
## **Explorative Data Analysis**

---
## **Feature Engineering**

---
## **Model Developement**

---
## **Model Deployment**